In [2]:
import numpy as np
import pandas as pd

In [37]:

def fix_timestamps(ds, format=None):
    if format is None:
        ds_fixed = pd.to_datetime(ds, format="%Y%m%d_%H%M%S")
    else:
        ds_fixed = pd.to_datetime(ds, format=format)
    return ds_fixed

def fix_locations(hdf):
    lat = pd.to_numeric(hdf['AR location'].str.strip().str.slice(start=0, stop=3).str.replace('N', '').str.replace('S', '-'))
    lon = pd.to_numeric(hdf['AR location'].str.strip().str.slice(start=3, stop=6).str.replace('W', '').str.replace('E', '-'))
    hdf['fl_lat'] = lat
    hdf['fl_lon'] = lon
    hdf['ar_location'] = 'POINT(' + lon.astype(str) + ' ' + lat.astype(str) + ')'
    return hdf

def get_hinode_flare_dataframe(file_path):
    """Reads the flare dataframe given in flare file path, downloaded using our script"""
    df = pd.read_csv(file_path, delimiter=',', index_col = 'Event number', parse_dates=True)
    
    hinode_timeformat = '%Y/%m%d %H:%M'
    df['end_time'] = fix_timestamps(df['end'], format='%Y-%m-%dT%H:%M:%S')
    df['start_time'] = fix_timestamps(df['start'], format='%Y-%m-%dT%H:%M:%S')
    df['peak_time'] = fix_timestamps(df['peak'], format='%Y-%m-%dT%H:%M:%S')
    df = fix_locations(df)
    df = append_hpc_coord(df)
    df.rename(columns={'X-ray class':'goes_class'}, inplace=True)
    df = df[['start_time', 'peak_time', 'end_time', 'goes_class', 'ar_location', 'fl_lat', 'fl_lon', 'x_hpc', 'y_hpc']]
    return df

import astropy.units as u
from astropy.coordinates import SkyCoord
from sunpy.coordinates import frames

def transform_fl_lat_lon(fdf):
    lat = fdf['fl_lat'] 
    lon = fdf['fl_lon']
    fdf['y'] = np.sin(np.deg2rad(lat))
    fdf['x'] = np.sin(np.deg2rad(lon)) * np.cos(np.deg2rad(lat))
    return fdf

def append_hpc_coord(fdf):
    fdf['x_hpc'] = fdf.apply(xcoord_transformer, axis=1) # x coordinates of hpc
    fdf['y_hpc'] = fdf.apply(ycoord_transformer, axis=1) # y coordinates of hpc
    return fdf

def xcoord_transformer(x):
    lon = x['fl_lon']
    lat = x['fl_lat']
    edate = x['start_time']
    c = SkyCoord(lon*u.deg, lat*u.deg, frame=frames.HeliographicStonyhurst, obstime=edate)
    c_hpc = c.transform_to(frames.Helioprojective)
    return c_hpc.Tx.arcsec #tx

def ycoord_transformer(x):
    lon = x['fl_lon']
    lat = x['fl_lat']
    edate = x['start_time']
    c = SkyCoord(lon*u.deg, lat*u.deg, frame=frames.HeliographicStonyhurst, obstime=edate)
    c_hpc = c.transform_to(frames.Helioprojective)
    return c_hpc.Ty.arcsec #Ty

In [38]:
hinode_path = '/home/baydin2/workspace/flarepredictiondata/flare_reading/datain/Hinode/Hinode_all.csv'
hdf = get_hinode_flare_dataframe(hinode_path)


hdf2010_2018 = hdf[hdf['start_time']>'2010-05-01']

hdf.to_csv('/home/baydin2/workspace/flarepredictiondata/flare_reading/datain/Hinode/Hinode_may2010_dec2018.csv', sep='\t')

In [39]:
hdf.head()



,start_time,peak_time,end_time,goes_class,ar_location,fl_lat,fl_lon,x_hpc,y_hpc
Event number,,,,,,,,,
165670,2018-12-29 13:16:00,2018-12-29 13:17:00,2018-12-29 13:18:00,A1.1,POINT(-89 -12),-12,-89,-953.860525,-201.777929
165660,2018-12-26 09:44:00,2018-12-26 09:46:00,2018-12-26 09:48:00,A2.1,POINT(-5 -50),-50,-5,-54.802026,-723.583858
165650,2018-12-20 00:01:00,2018-12-20 00:12:00,2018-12-20 00:21:00,A6.4,POINT(18 12),12,18,295.921168,227.444340
165640,2018-12-15 13:49:00,2018-12-15 13:53:00,2018-12-15 13:58:00,B1.0,POINT(-42 12),12,-42,-639.895684,214.996984
165630,2018-12-11 08:54:00,2018-12-11 09:00:00,2018-12-11 09:04:00,A3.1,POINT(-73 -1),-1,-73,-932.452715,-14.976526
